# Model Training

## **Setup**

In [ ]:
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.merge import merge
from scipy.spatial.distance import cdist
import numpy as np
import glob
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as cx
import folium
# import os

# --- Configuration ---
pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")


In [ ]:
# Get Point Reyes boundaries
# ================================================
cpad = gpd.read_file("../Data/CPAD_Release_2025b/CPAD_2025b_Units/CPAD_2025b_Units.shp")
pt_reyes = cpad[cpad['UNIT_NAME'] == 'Point Reyes National Seashore'].copy()
pt_reyes_boundary = pt_reyes.dissolve()
# Save
# pt_reyes_boundary.to_file("point_reyes_boundary.geojson", driver='GeoJSON')

# Plot
# ================================================
# Reproject to Web Mercator (EPSG:3857)
pt_reyes_web_mercator = pt_reyes_boundary.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 10))
pt_reyes_web_mercator.plot(ax=ax, alpha=1.0, facecolor='none', edgecolor='r', linewidth=2)
cx.add_basemap(ax, source=cx.providers.Esri.WorldTopoMap)
ax.set_axis_off()
plt.show()

# **Temporal Component - Weather conditions and mushroom emergence**

### Open-Meteo Data

Open-Meteo also has historical reanalysis data, but on a coarser 9km grid.

In [ ]:
reanalysis_om_df = pd.read_csv('../Data/weather_data_open-meteo/data/pt_reyes_weather_history_om.csv')
reanalysis_om_df.head()

reanalysis_om_df['date'] = pd.to_datetime(reanalysis_om_df['date'])
reanalysis_om_df['prcp_mm_7d_ma'] = reanalysis_om_df['prcp_mm'].rolling(window=7, min_periods=1).mean()

In [ ]:
import matplotlib.dates as mdates

fig, ax1 = plt.subplots(figsize=(16, 5))

ax1.bar(reanalysis_om_df['date'], reanalysis_om_df['prcp_mm'], color='blue', label='Precipitation (mm)', edgecolor='none')
ax1.set_ylabel('Precipitation (mm)', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
ax2.plot(reanalysis_om_df['date'], reanalysis_om_df['tmax_c'], color='red', linewidth=0.2, label='Tmax (°C)')
ax2.plot(reanalysis_om_df['date'], reanalysis_om_df['tmin_c'], color='teal', linewidth=0.2, label='Tmin (°C)')
ax2.set_ylabel('Temperature (°C)')
ax2.tick_params(axis='y')

# Optionally add a legend for temperature lines
lines, labels = ax2.get_legend_handles_labels()
ax2.legend(lines, labels, loc='upper right')

# Set x-ticks at each year and rotate labels 45 degrees for better readability
years = reanalysis_om_df['date'].dt.year.unique()
xticks = [reanalysis_om_df[reanalysis_om_df['date'].dt.year == y]['date'].iloc[0] for y in years]
ax1.set_xticks(xticks)
ax1.set_xticklabels(years, rotation=45) #, fontsize=12, color='black')

# Add prominent vertical grid lines at each year to further emphasize years
for xtick in xticks:
    ax1.axvline(x=xtick, color='gray', linestyle='--', linewidth=1, alpha=0.4, zorder=0)

# make horizontal grid lines thinner
ax1.grid(axis='x', which='both', linewidth=0)  # removes default vertical grid lines for ax1
ax1.grid(axis='y', which='both', linewidth=0.5, linestyle=':')
ax2.grid(axis='x', which='both', linewidth=0)  # removes default vertical grid lines for ax2
ax2.grid(axis='y', which='both', linewidth=0.5, linestyle=':')

# ax1.set_xlim(pd.Timestamp('2018-01-01'), pd.Timestamp('2020-12-31'))

plt.title('Temp and Precip Reanalysis Data (Open-Meteo), Bear Valley Visitor Center')
fig.tight_layout()
plt.show()

In [ ]:
# # Plot 7-day moving average of precipitation

# # Calculate 7-day moving average
# reanalysis_om_df['prcp_mm_7d_ma'] = reanalysis_om_df['prcp_mm'].rolling(window=7, min_periods=1).mean()

# fig = go.Figure()

# fig.add_trace(go.Scatter(
#     x=reanalysis_om_df['date'],
#     y=reanalysis_om_df['prcp_mm'],
#     name='Daily Precip',
#     mode='markers',
#     marker_color='green',
#     marker_size=5,
#     opacity=1.0
# ))

# fig.add_trace(go.Scatter(
#     x=reanalysis_om_df['date'],
#     y=reanalysis_om_df['prcp_mm_7d_ma'],
#     name='7-day MA',
#     mode='lines',
#     marker_color='green',
#     marker_size=5,
#     opacity=1.0
# ))

# fig.update_layout(
#     barmode='overlay',
#     title='Open Meteo precip data with 7-day moving average',
#     xaxis_title='Date',
#     yaxis_title='Precipitation (mm)',
#     legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
#     xaxis=dict(
#         rangeselector=dict(
#             buttons=list([
#                 dict(count=1, label="1y", step="year", stepmode="backward"),
#                 dict(count=3, label="3y", step="year", stepmode="backward"),
#                 dict(step="all")
#             ])
#         ),
#         rangeslider=dict(
#             visible=True
#         ),
#         type="date"
#     ),
#     height=400,
#     template='plotly_white'
# )

# fig.show()


# **Mushroom Sightings**

In [ ]:
# Load Mushroom Sightings from iNaturalist
print("Loading iNaturalist data...")
inat_df = pd.read_csv('../Data/inaturalist_data/fungi/observations-675943.csv/observations-675943.csv')

In [ ]:
# --- Define Target Species ---
CHOICE_EDIBLES = {
    # 'Chanterelle': ['Cantharellus californicus', 'Cantharellus formosus'],
    'King Bolete': ['Boletus edulis', 'Boletus edulis var. grandedulis'],
    # 'Candy Cap': ['Lactarius rubidus', 'Lactarius rufulus'],
    # 'Black Trumpet': ['Craterellus cornucopioides', 'Craterellus fallax'],
    # 'Hedgehog Mushroom': ['Hydnum repandum']
}

PROXY_SPECIES = {
    # Chanterelles are found with oaks, just like the highly visible Fly Agaric.
    # 'Chanterelle': [
    #     'Amanita muscaria', # Fly Agaric
    #     'Russula'           # Russula (Genus)
    # ],
    
    # King Boletes are pine-associates, sharing habitat with Suillus and Fly Agaric.
    'King Bolete': [
        'Suillus',          # Slippery Jacks (Genus)
        'Amanita muscaria'  # Fly Agaric
    ],

    # Black Trumpets like damp, mossy areas, similar to colorful Waxcaps and Coral Fungi.
    # 'Black Trumpet': [
    #     'Hygrocybe',        # Waxcaps (Genus)
    #     'Ramaria',          # Coral Fungi (Genus)
    #     'Clavaria'          # Coral Fungi (Genus)
    # ],

    # Candy Caps are a type of milk cap; other common milk caps indicate a suitable habitat.
    # 'Candy Cap': [
    #     'Lactarius alnicola' # A common, non-choice Milk Cap
    # ]
}

In [ ]:
# Filter the inaturalist data using scientific names in CHOICE_EDIBLES and PROXY_SPECIES
# (case insensitive, strip whitespace)
edible_sci_names = set(
    species.strip().lower()
    for species_list in CHOICE_EDIBLES.values()
    for species in species_list
)
proxy_sci_names = set(
    species.strip().lower()
    for species_list in PROXY_SPECIES.values()
    for species in species_list
)
all_target_sci_names = edible_sci_names | proxy_sci_names

edibles_df = inat_df[
    inat_df['scientific_name'].str.strip().str.lower().apply(
        lambda sci: any(target in sci for target in all_target_sci_names)
    )
].copy()

all_fungi_df = inat_df[inat_df['scientific_name'] != 'Fungi'].copy()



In [ ]:
all_fungi_df.value_counts('scientific_name', ascending=False)

In [ ]:
# Define search area as 2km buffer around Pt Reyes boundary
search_area = pt_reyes_boundary.to_crs(epsg=3857).buffer(2000)

# search_area_web_mercator = search_area.to_crs(epsg=3857)
fig, ax = plt.subplots(figsize=(10, 10))
search_area.plot(ax=ax, alpha=1.0, facecolor='none', edgecolor='r', linewidth=2)
cx.add_basemap(ax, source=cx.providers.Esri.WorldTopoMap)
ax.set_axis_off()
plt.show()

In [ ]:
# rf_model for sightings of target species in the rf_model area

# Invert both the CHOICE_EDIBLES and PROXY_SPECIES dictionaries for easy mapping from species name to group
# species_to_group = {}
# for group_dict in [CHOICE_EDIBLES, PROXY_SPECIES]:
#     for group, species_list in group_dict.items():
#         for species in species_list:
#             species_to_group[species] = group

# print(species_to_group)

# Filter for target species and add a 'group' column
# edibles_df = inat_df[inat_df['scientific_name'].isin(species_to_group.keys())].copy()
# edibles_df['group'] = edibles_df['scientific_name'].map(species_to_group)
edibles_df['observed_on'] = pd.to_datetime(edibles_df['observed_on'])
all_fungi_df['observed_on'] = pd.to_datetime(all_fungi_df['observed_on'])

# Add column "proxy_for" to edibles_df, which lists all choice edibles for which the row is a proxy species
EDIBLES_PROXIES_COMBINED = {}
for edible in CHOICE_EDIBLES.keys():
    EDIBLES_PROXIES_COMBINED[edible] = list(CHOICE_EDIBLES.get(edible, []))
    EDIBLES_PROXIES_COMBINED[edible].extend(PROXY_SPECIES.get(edible, []))

# print(EDIBLES_PROXIES_COMBINED)

edibles_df['proxy_for'] = edibles_df['scientific_name'].apply(
    lambda x: [edible for edible in CHOICE_EDIBLES.keys() if any(x.startswith(val) for val in EDIBLES_PROXIES_COMBINED[edible])]
    )

# print(edibles_df[['scientific_name', 'proxy_for']].head())

print(f"Found {len(edibles_df)} choice edible and proxy species sightings.")
print(f"Found {len(all_fungi_df)} total fungi sightings.")

# Convert sightings to a GeoDataFrame
edibles_gdf = gpd.GeoDataFrame(
    edibles_df, 
    geometry=gpd.points_from_xy(edibles_df.longitude, edibles_df.latitude),
    crs="EPSG:4326"
)
all_fungi_gdf = gpd.GeoDataFrame(
    all_fungi_df, 
    geometry=gpd.points_from_xy(all_fungi_df.longitude, all_fungi_df.latitude),
    crs="EPSG:4326"
)

# Filter sightings to Marin County
search_area_4326 = search_area.to_crs(edibles_gdf.crs)
pt_reyes_edibles_gdf = gpd.clip(edibles_gdf, search_area_4326)
pt_reyes_all_fungi_gdf = gpd.clip(all_fungi_gdf, search_area_4326)

print(f"Found {len(pt_reyes_edibles_gdf)} choice edible and proxy species sightings in Point Reyes region.")
print(f"Found {len(pt_reyes_all_fungi_gdf)} total fungi sightings in Point Reyes region.")

In [ ]:
# Plot all scientific names in the filtered iNaturalist edibles data with at least 20 sightings (most common first)
# import re

order_counts = pt_reyes_edibles_gdf['scientific_name'].value_counts()
order_counts = order_counts[order_counts >= 20]

plot_df = (
    pt_reyes_edibles_gdf[pt_reyes_edibles_gdf['scientific_name'].isin(order_counts.index)]
    .groupby('scientific_name')
    .size()
    .reset_index(name='count')
    .sort_values(by='count', ascending=False)
)

plt.figure(figsize=(12, max(8, 0.5 * len(order_counts))))  # Make figure taller based on number of names

ax = sns.barplot(
    data=plot_df,
    y='scientific_name',
    x='count',
    order=plot_df['scientific_name'],  # ensures identical order
    color='skyblue',
    linewidth=0,
    orient='h',
)

plt.title('Sightings by Scientific Name (Choice Edibles in Bold)')
plt.xlabel('Number of Sightings')

# --- Make y-axis label bold for any choice edible scientific names ---
# Build a flat set of all edible scientific names (lowercased & stripped) from the CHOICE_EDIBLES dict
edibles_sci_names = set(
    species.strip().lower()
    for species_list in CHOICE_EDIBLES.values()
    for species in species_list
)
# Get all tick labels, compare (case-insensitive) if ANY edible name is a substring, and set fontweight
for label in ax.get_yticklabels():
    sci_name = label.get_text().strip().lower()
    if any(edible in sci_name for edible in edibles_sci_names):
        label.set_fontweight('bold')

plt.ylabel('Scientific Name')
plt.tight_layout()
plt.show()

## Sighting Map of Point Reyes region

In [ ]:
# Fix: Preserve original point geometries
fig, ax = plt.subplots(figsize=(15, 15))

# Plot Marin boundary for context
pt_reyes_boundary_web = pt_reyes_boundary.to_crs(epsg=3857)
pt_reyes_boundary_web.plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1)

# Use the original pt_reyes_edibles_gdf (which has Point geometries) instead of enriched_gdf
plot_gdf = pt_reyes_edibles_gdf.to_crs(epsg=3857)  # Use original points

# Only keep rows where geometry is Point
points_only = plot_gdf[plot_gdf.geometry.geom_type == "Point"].copy()


# Extract x/y for scatterplot
points_only["x"] = points_only.geometry.apply(lambda geom: geom.x)
points_only["y"] = points_only.geometry.apply(lambda geom: geom.y)

# Get unique groups and colors
edibles = CHOICE_EDIBLES.keys()
colors = sns.color_palette("Set2", n_colors=len(edibles))

# Plot each choice edible group separately with edge colors
for i, edible in enumerate(edibles):
    # edible_data = points_only[points_only["proxy_for"].apply(lambda proxies: isinstance(proxies, list) and "edible" in proxies)]
    edible_data = points_only[points_only["proxy_for"].apply(lambda proxy_edibles: edible in proxy_edibles)]
    # print(edible_data.head())
    
    # Create darker edge color
    edge_color = tuple([max(c * 0.6, 0) for c in colors[i]])
    
    ax.scatter(
        edible_data["x"], 
        edible_data["y"], 
        c=[colors[i]], 
        s=50, 
        alpha=0.8,
        linewidth=0.8,
        edgecolor=edge_color,
        label=edible
    )

# Add basemap
cx.add_basemap(ax, source=cx.providers.Esri.WorldTopoMap, zoom=13)

ax.set_title('Porcini/Proxy Sightings in Point Reyes', fontsize=16, fontweight='bold')
ax.set_axis_off()
plt.legend(title='Species Group')
plt.show()

print(f"Plotted {len(points_only)} mushroom sightings")

## Sightings over time

In [ ]:
print(pt_reyes_edibles_gdf.columns)
pt_reyes_edibles_gdf.head()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots



# Count reports by date
reports_by_date = pt_reyes_edibles_gdf.groupby('observed_on').size().reset_index(name='count')
# Our weather data starts in 2010 - remove any sightings before then (**iNaturalist was create in 2008, first app was launched in 2011**)
reports_by_date = reports_by_date[reports_by_date['observed_on'] >= pd.Timestamp('2010-01-01')]
print(reports_by_date.head())

# Ensure date columns are parsed as datetime
reports_by_date['observed_on'] = pd.to_datetime(reports_by_date['observed_on'])

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])


# Plot Precipitation (mm) - Open-Meteo
fig.add_trace(go.Scatter(
    x=reanalysis_om_df['date'],
    y=reanalysis_om_df['prcp_mm'],
    name='Daily Precip',
    mode='markers',
    marker_color='green',
    marker_size=1,
    opacity=1.0
))

# Plot Precipitation (mm) - Open-Meteo 7-day moving average
fig.add_trace(go.Scatter(
    x=reanalysis_om_df['date'],
    y=reanalysis_om_df['prcp_mm_7d_ma'],
    name='Precip 7-day MA',
    mode='lines',
    marker_color='green',
    line_width=1,
    opacity=1.0
))

# Plot mushroom observation counts on a secondary y-axis (right)
fig.add_trace(go.Bar(
    x=reports_by_date['observed_on'],
    y=reports_by_date['count'],
    name='Mushroom Sightings',
    marker_color='brown',
    opacity=1.0
    ),
    secondary_y=True
)


fig.update_layout(
    barmode='overlay',
    title='Mushroom Sightings by Date, with Precipitation',
    xaxis_title='Date',
    yaxis_title='Number of Sightings',
    legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=3, label="3y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    height=400,
    template='plotly_white'
)

fig.show()

In [ ]:
pt_reyes_edibles_by_date = pt_reyes_edibles_gdf.groupby('observed_on').agg(
    unique_users_edibles = ('user_id', 'nunique'),
    obs_edibles = ('id','nunique')
).fillna(0)
pt_reyes_all_fungi_by_date = pt_reyes_all_fungi_gdf.groupby('observed_on').agg(
    unique_users_fungi = ('user_id', 'nunique'),
    obs_fungi = ('id','nunique')
).fillna(0)

pt_reyes_obs_by_date = pt_reyes_edibles_by_date.merge(pt_reyes_all_fungi_by_date, left_on='observed_on', right_on='observed_on', how='outer').reset_index().fillna(0)

plt.scatter(pt_reyes_obs_by_date['unique_users_fungi'], pt_reyes_obs_by_date['unique_users_edibles'], alpha=0.5, s=5)
plt.xlabel('Count Unique Users w Fungi')
plt.ylabel('Count Unique Users w Edibles')
plt.title('Unique Users w Fungi and Edibles by Date')
plt.show()


In [ ]:
pt_reyes_obs_by_date.head()

In [ ]:
pt_reyes_obs_by_date['unique_users_edibles_smoothed'] = pt_reyes_obs_by_date['unique_users_edibles'].rolling(window=7, win_type='gaussian', center=True).mean(std=2)
pt_reyes_obs_by_date['unique_users_fungi_smoothed'] = pt_reyes_obs_by_date['unique_users_fungi'].rolling(window=7, win_type='gaussian', center=True).mean(std=2)

In [ ]:
# plt.plot(pt_reyes_obs_by_date['observed_on'], pt_reyes_obs_by_date['unique_users_edibles'], 'bo', markersize=2, label='Unique Users Edibles')
# plt.plot(pt_reyes_obs_by_date['observed_on'], pt_reyes_obs_by_date['unique_users_edibles_smoothed'], 'b-', label='Unique Users Edibles Smoothed')
# plt.plot(pt_reyes_obs_by_date['observed_on'], pt_reyes_obs_by_date['unique_users_fungi'], 'ro', markersize=2, label='Unique Users Fungi')
# plt.plot(pt_reyes_obs_by_date['observed_on'], pt_reyes_obs_by_date['unique_users_fungi_smoothed'], 'r-', label='Unique Users Fungi Smoothed')
# plt.xlim(pd.Timestamp('2025-12-01'), pd.Timestamp('2025-12-31'))
# plt.legend()
# plt.show()

## Engineer Features

### Merge OpenMeteo and iNaturalist datasets

In [ ]:
# model_data = reports_by_date.merge(reanalysis_om_df, left_on='observed_on', right_on='date', how='outer')
# model_data['count'] = model_data['count'].fillna(0)
model_data = pt_reyes_obs_by_date.merge(reanalysis_om_df, left_on='observed_on', right_on='date', how='right')
model_data['unique_users_edibles'] = model_data['unique_users_edibles'].fillna(0)
model_data['obs_edibles'] = model_data['obs_edibles'].fillna(0)
model_data['unique_users_fungi'] = model_data['unique_users_fungi'].fillna(0)
model_data['obs_fungi'] = model_data['obs_fungi'].fillna(0)
model_data.head(20)


#### Smooth mushroom observation data

In [ ]:
model_data['unique_users_edibles_smoothed'] = model_data['unique_users_edibles'].rolling(window=7, win_type='gaussian', center=True).mean(std=2)
model_data['unique_users_fungi_smoothed'] = model_data['unique_users_fungi'].rolling(window=7, win_type='gaussian', center=True).mean(std=2)

#### Weather features - range of rolling Gaussian windows

In [ ]:
# 14-day precipitation (simple moving sum)
model_data['14_day_prcp_mm'] = model_data['prcp_mm'].rolling(window=14, closed='left').sum()
# 30-day precipitation (simple moving sum)
model_data['30_day_prcp_mm'] = model_data['prcp_mm'].rolling(window=30, closed='left').sum()
# 60-day precipitation (simple moving sum)
model_data['60_day_prcp_mm'] = model_data['prcp_mm'].rolling(window=60, closed='left').sum()

# Exponential moving averages with half-lives of 7, 14, 30, and 60 days
model_data['prcp_mm_ema_hl7'] = model_data['prcp_mm'].ewm(halflife=7, adjust=False).mean()
model_data['prcp_mm_ema_hl14'] = model_data['prcp_mm'].ewm(halflife=14, adjust=False).mean()
model_data['prcp_mm_ema_hl30'] = model_data['prcp_mm'].ewm(halflife=30, adjust=False).mean()
model_data['prcp_mm_ema_hl60'] = model_data['prcp_mm'].ewm(halflife=60, adjust=False).mean()

# model_data['14_day_prcp_mm'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# plt.xlabel('14-day Precipitation (mm)')
# plt.show()


In [ ]:
# Create columns for various rolling sums, using gaussian window

windows = np.arange(5, 61, 5)
std_devs = np.arange(2, 12, 3)
results = np.zeros((len(windows), len(std_devs)))
for i, w in enumerate(windows):
    for j, std in enumerate(std_devs):
        prcp_col_string = f'prcp_mm_gauss_{w}d_{std}std'
        model_data[prcp_col_string] = model_data['prcp_mm'].rolling(
            window=w, 
            win_type='gaussian'
        ).sum(std=std).shift(1) # shift(1) to include only prior days (not today)


windows = np.arange(5, 31, 5)
std_devs = np.arange(2, 9, 3)
results = np.zeros((len(windows), len(std_devs)))
for i, w in enumerate(windows):
    for j, std in enumerate(std_devs):
        tmax_col_string = f'tmax_c_gauss_{w}d_{std}std'
        model_data[tmax_col_string] = model_data['tmax_c'].rolling(
            window=w, 
            win_type='gaussian'
        ).mean(std=std).shift(1)

        tmin_col_string = f'tmin_c_gauss_{w}d_{std}std'
        model_data[tmin_col_string] = model_data['tmin_c'].rolling(
            window=w, 
            win_type='gaussian'
        ).mean(std=std).shift(1)
        
        
        
        
        

In [ ]:
# model_data.columns

In [ ]:
# model_data['7day_tmax_c'] = model_data['tmax_c'].rolling(window=7, closed='left').mean()
# # model_data['7day_tmax_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# # plt.xlabel('7-day Moving Average of Max Temp (C)')
# # plt.show()

# model_data['7day_tmin_c'] = model_data['tmin_c'].rolling(window=7, closed='left').mean()
# # model_data['7day_tmin_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# # plt.xlabel('7-day Moving Average of Min Temp (C)')
# # plt.show()



In [ ]:
# model_data['14day_tmax_c'] = model_data['tmax_c'].rolling(window=14, closed='left').mean()
# # model_data['14day_tmax_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# # plt.xlabel('14-day Moving Average of Max Temp (C)')
# # plt.show()

# model_data['14day_tmin_c'] = model_data['tmin_c'].rolling(window=14, closed='left').mean()
# # model_data['14day_tmin_c'].plot.hist(bins=20, weights=model_data['count']) #, edgecolor='black')
# # plt.xlabel('14-day Moving Average of Min Temp (C)')
# # plt.show()

# model_data['14-7_tmax_c'] = model_data['14day_tmax_c']-model_data['7day_tmax_c']
# # model_data['14-7_tmax_c'].plot.hist(bins=10, weights=model_data['count']) #, edgecolor='black')
# # plt.xlabel('14-day Moving Average of Max Temp (C) - 7-day Moving Average of Max Temp (C)')
# # plt.show()

# model_data['14-7_tmin_c'] = model_data['14day_tmin_c']-model_data['7day_tmin_c']
# # model_data['14-7_tmin_c'].plot.hist(bins=10, weights=model_data['count']) #, edgecolor='black')
# # plt.xlabel('14-day Moving Average of Min Temp (C) - 7-day Moving Average of Min Temp (C)')
# # plt.show()



In [ ]:
model_data['dayofweek'] = model_data['date'].dt.dayofweek

model_data['is_weekend'] = model_data['dayofweek'].isin([5,6])
model_data['is_weekend'].value_counts()

In [ ]:
# Include the day of year in sin/cos time
model_data['sin_time'] = np.sin(2 * np.pi * model_data['yday'] / 365)
model_data['cos_time'] = np.cos(2 * np.pi * model_data['yday'] / 365)

# plt.plot(model_data['yday'], model_data['sin_time'], 'r-', label='sin_time')
# plt.plot(model_data['yday'], model_data['cos_time'], 'b-', label='cos_time')
# plt.legend()
# plt.show()


In [ ]:
print(model_data.tail())

In [ ]:
# Filter to 2016-2026 (more iNaturalist data, end on an even year for time series split)
start_year = 2016
end_year = 2026

start_date = f'{start_year}-01-01'
end_date = f'{end_year-1}-12-31'

model_data_filtered = model_data[(model_data['date'] >= start_date) & (model_data['date'] <= end_date)].copy()

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

# Define a time series split for all cross-validation
test_years = 2 # Leave a couple years for testing
# cv = TimeSeriesSplit(n_splits=(end_year-start_year-test_years))

# Mark the training and testing period
split_idx = int(len(model_data_filtered) * (1 - test_years/(end_year-start_year)))
model_data_filtered['train_test'] = 'train'
model_data_filtered.iloc[split_idx:, model_data_filtered.columns.get_loc('train_test')] = 'test'

In [ ]:
# model_data_filtered.head()

## Model training

#### Setup - reduce features, train/test split, define sample weights

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

prcp_cols = model_data.filter(regex=r'^prcp_mm_gauss_').columns.tolist()
tmax_cols = model_data.filter(regex=r'^tmax_c_gauss_').columns.tolist()
tmin_cols = model_data.filter(regex=r'^tmin_c_gauss_').columns.tolist()
time_cols = ['sin_time', 'cos_time']
# or: model_data.columns[model_data.columns.str.startswith('prcp_mm_gl_')].tolist()

model_features = prcp_cols + tmax_cols + tmin_cols + time_cols 
# + [
#     # # observer effort features
#     # 'tmax_c', 'tmin_c', 'prcp_mm', 'is_weekend', #'dayofweek',
#     # precip features
#     # '14_day_prcp_mm', '30_day_prcp_mm', '60_day_prcp_mm',     
#     # 'prcp_mm_ema_hl7','prcp_mm_ema_hl14','prcp_mm_ema_hl30','prcp_mm_ema_hl60',
#     # 'prcp_mm_gl_7d','prcp_mm_gl_14d','prcp_mm_gl_30d','prcp_mm_gl_60d',
#     # 'days_since_last_precip_over_1mm', 'days_since_last_precip_over_3mm', 
#     # 'days_since_last_precip_over_5mm',
#     # temperature features
#     # '7day_tmax_c', '7day_tmin_c', '14day_tmax_c', '14day_tmin_c', '14-7_tmax_c', '14-7_tmin_c',
#     # time features (exclude - mushrooms don't know the date)
#     'sin_time', 'cos_time'
# ]

# X = model_data_filtered[model_features].copy()
# y = model_data_filtered['unique_users_edibles_smoothed'].copy()
# w = np.log1p(model_data_filtered['unique_users_fungi_smoothed'].copy()) + 1 # Add 1 to give weight to zero days

# X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(X, y, w, test_size=(test_years/(end_year-start_year)), shuffle=False)

X_train = model_data_filtered.loc[(model_data_filtered['train_test'] == 'train'), model_features].copy()
X_test = model_data_filtered.loc[(model_data_filtered['train_test'] == 'test'), model_features].copy()
y_train = model_data_filtered.loc[(model_data_filtered['train_test'] == 'train'), 'unique_users_edibles_smoothed'].copy()
y_test = model_data_filtered.loc[(model_data_filtered['train_test'] == 'test'), 'unique_users_edibles_smoothed'].copy()
w_train = model_data_filtered.loc[(model_data_filtered['train_test'] == 'train'), 'unique_users_fungi_smoothed'].copy()
w_test = model_data_filtered.loc[(model_data_filtered['train_test'] == 'test'), 'unique_users_fungi_smoothed'].copy()

In [ ]:
print(np.max(w_test))
print(np.max(w_train))
print(np.max(w))
print(np.min(w))
print("Number of NaNs in w:", w.isna().sum())
# print(test_years/(end_year-start_year))
# print(model_data_filtered['unique_users_fungi'].value_counts())

print("Number of NaNs in y:", y.isna().sum())
print(model_data_filtered.loc[model_data_filtered['unique_users_edibles_smoothed'].isna(), 'date'])

In [ ]:
import pandas as pd
import numpy as np
import scipy.cluster.hierarchy as hc
from scipy.stats import spearmanr

# Calculate the Spearman rank correlation matrix of features
corr_matrix, _ = spearmanr(X)
corr_linkage = hc.ward(corr_matrix)

# Group features into clusters (e.g., setting a threshold of 0.85 similarity)
# Any features with > 0.85 correlation will be forced into the same cluster
cluster_ids = hc.fcluster(corr_linkage, t=0.85, criterion='distance')
cluster_dict = {}
for i, cluster_id in enumerate(cluster_ids):
    feature_name = X.columns[i]
    if cluster_id not in cluster_dict:
        cluster_dict[cluster_id] = []
    cluster_dict[cluster_id].append(feature_name)

# Select the best feature from each cluster
final_features = []
for cluster, features in cluster_dict.items():
    if len(features) == 1:
        final_features.append(features[0])
    else:
        # Find which feature in the cluster correlates best with your target (y)
        best_feature = None
        best_corr = -1
        
        for feature in features:
            # Drop NaNs to calculate correlation safely
            temp_df = pd.DataFrame({'feature': X[feature], 'target': y}).dropna()
            target_corr = np.abs(temp_df['feature'].corr(temp_df['target']))
            
            if target_corr > best_corr:
                best_corr = target_corr
                best_feature = feature
                
        final_features.append(best_feature)

print(f"Reduced from {len(X.columns)} to {len(final_features)} independent features.")
print(final_features)

In [ ]:
X_train = model_data_filtered.loc[(model_data_filtered['train_test'] == 'train'), final_features].copy()
X_test = model_data_filtered.loc[(model_data_filtered['train_test'] == 'test'), final_features].copy()
y_train = model_data_filtered.loc[(model_data_filtered['train_test'] == 'train'), 'unique_users_edibles_smoothed'].copy()
y_test = model_data_filtered.loc[(model_data_filtered['train_test'] == 'test'), 'unique_users_edibles_smoothed'].copy()
w_train = model_data_filtered.loc[(model_data_filtered['train_test'] == 'train'), 'unique_users_fungi_smoothed'].copy()
w_test = model_data_filtered.loc[(model_data_filtered['train_test'] == 'test'), 'unique_users_fungi_smoothed'].copy()

#### Random Forest model

In [ ]:
import optuna
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, KFold
from sklearn.model_selection import KFold

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 20, 160, step=20),
        'max_depth': trial.suggest_int('max_depth', 2, 8),
        'min_samples_split': trial.suggest_int('min_samples_split', 8, 16),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 4, 16),
        'max_features': trial.suggest_int('max_features', 1, 10),
        'n_jobs': -1,
        'random_state': 42
    }

    # rf_model = RandomForestRegressor(**params)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_errors = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
        w_tr, w_va = w_train.iloc[train_idx], w_train.iloc[val_idx]

        rf = RandomForestRegressor(**params)
        rf.fit(X_tr, y_tr, sample_weight=w_tr)

        preds = rf.predict(X_va)
        val_mae = mean_absolute_error(y_va, preds, sample_weight=w_va)
        fold_errors.append(val_mae)

    return np.mean(fold_errors)

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

best_params = study.best_params
print(f"Best parameters found: {best_params}")

# Train final model with the best found hyperparameters
best_rf = RandomForestRegressor(
    n_estimators=best_params['n_estimators'],
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    n_jobs=-1,
    random_state=42
)

best_rf.fit(X_train, y_train)

y_pred = best_rf.predict(X_test)

mae = mean_absolute_error(y_test, y_pred, sample_weight=w_test)
mse = mean_squared_error(y_test, y_pred, sample_weight=w_test)
r2 = r2_score(y_test, y_pred, sample_weight=w_test)

print(f"MAE: {mae}")
print(f"MSE: {mse}")
print(f"R2: {r2}")

plt.scatter(y_test, y_pred)
plt.xlabel('Actual Count')
plt.ylabel('Predicted Count')
plt.title('Actual vs Predicted Counts')
plt.show()

In [ ]:
# Best parameters found: {'n_estimators': 40, 'max_depth': 8, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 10}
# MAE: 0.3587038928249882
# MSE: 0.4168995362759342
# R2: 0.2730638165345458

# Best parameters found: {'n_estimators': 140, 'max_depth': 8, 'min_samples_split': 12, 'min_samples_leaf': 4, 'max_features': 7}
# MAE: 0.40218796619129665
# MSE: 0.4801666624982762
# R2: 0.16274667949542898

In [ ]:
model_data_filtered["pred_count_rf"] = best_rf.predict(X[final_features])

In [ ]:
import numpy as np

# Get feature importances and columns
importances = best_rf.feature_importances_
columns = X[final_features].columns

# Get indices for top 20 features
top_indices = np.argsort(importances)[-20:][::-1]
top_features = columns[top_indices]
top_importances = importances[top_indices]

# print("Top 20 feature importances:")
# for feat, imp in zip(top_features, top_importances):
#     print(f"{feat}: {imp}")

# Plot the top 20 feature importances
plt.figure(figsize=(10, 6))
plt.bar(top_features, top_importances)
plt.xlabel('Top 20 Features')
plt.ylabel('Importance')
plt.xticks(rotation=90)
plt.title('Top 20 Feature Importances')
plt.tight_layout()
plt.show()

In [ ]:
print(y)

#### XGBoost model

In [ ]:
import numpy as np
import optuna
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor


def objective(trial):
    params = {
        'objective': 'reg:absoluteerror',
        'eval_metric': 'mae',
        'tree_method': 'hist',
        'learning_rate': trial.suggest_float(
            'learning_rate', 0.01, 0.1, log=True
        ),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 50.0, log=True),
        'n_estimators': 1500,
        'early_stopping_rounds': 30,
        'random_state': 42,
    }

    # Split by Year so entire seasons are held out
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_errors = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
        w_tr, w_va = w_train.iloc[train_idx], w_train.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)],
            sample_weight_eval_set=[w_va],
            verbose=False,
        )

        preds = model.predict(X_va)
        mae = mean_absolute_error(y_va, preds, sample_weight=w_va)
        fold_errors.append(mae)

    return np.mean(fold_errors)


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best parameters for XGBoost:", study.best_params)

best_xgb_model = XGBRegressor(
    **study.best_params,
    random_state=42,
    objective='reg:squarederror',
    eval_metric='rmse'
)
best_xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    # early_stopping_rounds=10,
    verbose=False
)

y_pred_xgb = best_xgb_model.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb, sample_weight=w_test)
mse_xgb = mean_squared_error(y_test, y_pred_xgb, sample_weight=w_test)
r2_xgb = r2_score(y_test, y_pred_xgb, sample_weight=w_test)

print(f"XGBoost MAE: {mae_xgb}")
print(f"XGBoost MSE: {mse_xgb}")
print(f"XGBoost R2: {r2_xgb}")

plt.scatter(y_test, y_pred_xgb)
plt.xlabel('Actual Count')
plt.ylabel('Predicted XGBoost Count')
plt.title('Actual vs Predicted Counts (XGBoost)')
plt.show()

In [ ]:
# Best parameters for XGBoost: {'learning_rate': 0.04016170068762172, 'max_depth': 7, 'min_child_weight': 11, 'subsample': 0.7980476723913897, 'colsample_bytree': 0.7643609853943039, 'reg_alpha': 0.5802388524872822, 'reg_lambda': 4.355138384427838}
# XGBoost MAE: 0.32072411754816743
# XGBoost MSE: 0.3361862523418743
# XGBoost R2: 0.4138013359429501

In [ ]:
import xgboost as xgb

xgb.plot_importance(best_xgb_model, importance_type='gain', height=0.4, max_num_features=20)
plt.title('Top 20 Feature Importances (Gain)')
plt.show()

## Model evaluation

In [ ]:
model_data_filtered['users_edibles_v_fungi'] = model_data_filtered['unique_users_edibles'] / model_data_filtered['unique_users_fungi']

# model_data_filtered['users_edibles_v_fungi'].value_counts()

# num_nan_users_edibles_v_fungi = model_data_filtered['users_edibles_v_fungi'].isna().sum()
# print(f"Number of NaN values in users_edibles_v_fungi: {num_nan_users_edibles_v_fungi}")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])


# Plot Precipitation (mm) - Open-Meteo
fig.add_trace(go.Bar(
    x=model_data_filtered['date'],
    y=model_data_filtered['prcp_mm'],
    name='Precipitation (mm) - Open-Meteo',
    # mode='markers',
    # marker_color='blue',
    # marker_size=1,
    opacity=1.0
))

# # Plot user counts on a secondary y-axis (right)
# model_data_filtered = model_data_filtered.sort_values(by='observed_on')
# fig.add_trace(go.Scatter(
#     x=model_data_filtered['date'],
#     y=model_data_filtered['unique_users_edibles'],
#     name='Users Reporting Edibles',
#     mode='markers',
#     marker_color='brown',
#     marker_size=5,
#     marker_symbol='circle',
#     opacity=1.0
#     ),
#     secondary_y=True
# )

# # Plot total users reporting fungi (observer effort)
# fig.add_trace(go.Scatter(
#     x=predX_rf['date'],
#     y=predX_rf['unique_users_fungi'],
#     name='Total Users Reporting Fungi (effort)',
#     mode='markers',
#     marker_color='brown',
#     marker_size=4,
#     marker_symbol='circle',
#     opacity=0.5
#     ),
#     secondary_y=True
# )

# Plot user counts edibles/fungion a secondary y-axis (right)
# model_data_filtered = model_data_filtered.sort_values(by='observed_on')
# fig.add_trace(go.Scatter(
#     x=model_data_filtered['date'],
#     y=model_data_filtered['unique_users_edibles'].rolling(window=7, center=True).mean(),
#     name='Users Reporting Edibles Rolling 7d',
#     mode='lines',
#     marker_color='brown',
#     # marker_size=5,
#     marker_symbol='circle',
#     opacity=1.0
#     ),
#     secondary_y=True
# )
fig.add_trace(go.Scatter(
    x=model_data_filtered['date'],
    y=model_data_filtered['unique_users_edibles_smoothed'],
    name='Users Reporting Edibles',
    mode='markers',
    marker_color='brown',
    marker_size=3,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

# Plot user counts from RF model
predX_rf = model_data_filtered.copy()
predX_rf['pred_users_rf'] = best_rf.predict(predX_rf[final_features])
predX_rf = predX_rf.sort_values(by='date')
fig.add_trace(go.Scatter(
    x=predX_rf['date'],
    y=predX_rf['pred_users_rf'],
    name='RF Predicted Users Reporting Edibles',
    mode='lines+markers',
    marker_color='green',
    marker_size=4,
    line_width=1,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

# Plot user counts from XGBoost model
predX_xgb = model_data_filtered.copy()
predX_xgb['pred_users_xgb'] = best_xgb_model.predict(predX_xgb[final_features])
predX_xgb = predX_xgb.sort_values(by='date')
fig.add_trace(go.Scatter(
    x=predX_xgb['date'],
    y=predX_xgb['pred_users_xgb'],
    name='XGB Predicted Users Reporting Edibles',
    mode='lines+markers',
    marker_color='red',
    marker_size=4,
    line_width=1,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

# print(predX['observed_on'].min())

# Plot predicted mushroom observation counts under ideal conditions (weekend, good weather)
# predX_ideal = predX.copy()
# predX_ideal['is_weekend'] = True
# predX_ideal['tmax_c'] = 12
# predX_ideal['tmin_c'] = 8
# predX_ideal['prcp_mm'] = 0
# predX_ideal['predicted_count'] = rf_model.predict(predX_ideal[model_features])
# predX_ideal = predX_ideal.sort_values(by='date')
# fig.add_trace(go.Scatter(
#     x=predX_ideal['date'],
#     y=predX_ideal['predicted_count'],
#     name='Predicted Sightings (Ideal)',
#     mode='lines+markers',
#     marker_color='blue',
#     marker_size=4,
#     line_width=1,
#     marker_symbol='circle',
#     opacity=1.0
#     ),
#     secondary_y=True
# )

fig.update_layout(
    barmode='overlay',
    title='Num Users Reporting Edibles and Predictions by Date, with Precipitation',
    xaxis_title='Date',
    yaxis_title='Precip (mm)',
    yaxis2_title='Count users',
    legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=3, label="3y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    height=800,
    template='plotly_white'
)

fig.show()

In [ ]:
# fig, ax = plt.subplots(figsize=(16, 4))

# ax.plot(predX_rf['cos_time'], predX_rf['predicted_count'], linewidth=1)
# ax.plot(predX_xgb['cos_time'], predX_xgb['predicted_count'], linewidth=1, color='red')
# # ax.plot([366-182, 366-182], [0, 8], 'k--', linewidth=1, label='Jan 1st')
# ax.set_xlabel('Cosine Time')
# ax.set_ylabel('Predicted Count')
# ax.set_title('Predicted Count by Cosine Time')
# ax.legend()
# plt.show()


In [ ]:
# # Save the best model
# import pickle

# # Save the best model to a file
# import datetime
# current_date = datetime.datetime.now().strftime('%Y-%m-%d')
# with open(f'model_{current_date}.pkl', 'wb') as f:
#     pickle.dump(rf_model.best_estimator_, f)


In [ ]:
# predX[['date'] + model_features].to_csv('predX.csv', index=False)

## Two-stage approach

1) Binary classification fruiting/no fruiting
2) If fruiting, predict abundance

In [ ]:
model_data_filtered.columns

In [ ]:
model_data_filtered['fruiting'] = (model_data_filtered['unique_users_edibles_smoothed'] > 0) * 1
model_data_filtered['fruiting'].value_counts()

# model_data_filtered['fruiting'].isna().sum()


In [ ]:
# X = model_data_filtered[model_features].copy()
# # X['ones'] = 1
# y = model_data_filtered['fruiting'].copy()
# w = model_data_filtered['unique_users_fungi_smoothed'].copy()

# X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(X, y, w, test_size=(test_years/(end_year-start_year)), shuffle=False)

X_train = model_data_filtered.loc[(model_data_filtered['train_test'] == 'train'), final_features].copy()
X_test = model_data_filtered.loc[(model_data_filtered['train_test'] == 'test'), final_features].copy()
y_train = model_data_filtered.loc[(model_data_filtered['train_test'] == 'train'), 'fruiting'].copy()
y_test = model_data_filtered.loc[(model_data_filtered['train_test'] == 'test'), 'fruiting'].copy()
w_train = model_data_filtered.loc[(model_data_filtered['train_test'] == 'train'), 'unique_users_fungi_smoothed'].copy()
w_test = model_data_filtered.loc[(model_data_filtered['train_test'] == 'test'), 'unique_users_fungi_smoothed'].copy()

In [ ]:
import numpy as np
import optuna
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor


def objective(trial):
    params = {
        'objective': 'reg:absoluteerror',
        'eval_metric': 'mae',
        'tree_method': 'hist',
        'learning_rate': trial.suggest_float(
            'learning_rate', 0.01, 0.1, log=True
        ),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 50.0, log=True),
        'n_estimators': 1500,
        'early_stopping_rounds': 30,
        'random_state': 42,
    }

    # Split by Year so entire seasons are held out
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_errors = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
        w_tr, w_va = w_train.iloc[train_idx], w_train.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)],
            sample_weight_eval_set=[w_va],
            verbose=False,
        )

        preds = model.predict(X_va)
        mae = mean_absolute_error(y_va, preds, sample_weight=w_va)
        fold_errors.append(mae)

    return np.mean(fold_errors)


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best parameters for XGBoost:", study.best_params)

best_xgb_classifier = XGBRegressor(
    **study.best_params,
    random_state=42,
    objective='reg:squarederror',
    eval_metric='rmse'
)
best_xgb_classifier.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    # early_stopping_rounds=10,
    verbose=False
)

y_pred_xgb = best_xgb_classifier.predict(X_test)

# mae_xgb = mean_absolute_error(y_test, y_pred_xgb, sample_weight=w_test)
# mse_xgb = mean_squared_error(y_test, y_pred_xgb, sample_weight=w_test)
# r2_xgb = r2_score(y_test, y_pred_xgb, sample_weight=w_test)

# print(f"XGBoost MAE: {mae_xgb}")
# print(f"XGBoost MSE: {mse_xgb}")
# print(f"XGBoost R2: {r2_xgb}")


In [ ]:

from sklearn.metrics import precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt

# For regression, we'll need to binarize the target (e.g., treat count > 0 as positive class)
y_test_bin = (y_test > 0).astype(int)
y_score = y_pred_xgb  # regression scores

precision, recall, _ = precision_recall_curve(y_test_bin, y_score)
ap_score = average_precision_score(y_test_bin, y_score)

plt.figure(figsize=(8,6))
plt.plot(recall, precision, label=f'Precision-Recall curve (AP = {ap_score:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve (XGBoost)')
plt.legend()
plt.show()


# plt.scatter(y_test, y_pred_xgb)
# plt.xlabel('Actual Count')
# plt.ylabel('Predicted XGBoost Count')
# plt.title('Actual vs Predicted Counts (XGBoost)')
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

# Get predicted probabilities for the positive class (fruiting) on your validation/test set
# y_prob = stage1_classifier.predict_proba(X_test)[:, 1]

# Generate precision, recall, and thresholds
precisions, recalls, thresholds = precision_recall_curve(y_test_bin, y_score)

# Calculate F2 score for each threshold (beta=2)
# Formula: (1 + beta^2) * (Precision * Recall) / ((beta^2 * Precision) + Recall)
f2_scores = (5 * precisions[:-1] * recalls[:-1]) / (4 * precisions[:-1] + recalls[:-1])

# Find the threshold that maximizes the F2 score
optimal_idx = np.argmax(f2_scores)
optimal_p = thresholds[optimal_idx]

print(f"Optimal Threshold (Max F2): {optimal_p:.4f}")
print(f"Precision at this threshold: {precisions[optimal_idx]:.4f}")
print(f"Recall at this threshold: {recalls[optimal_idx]:.4f}")


plt.figure(figsize=(10, 5))

# Plot Precision and Recall across thresholds
plt.plot(thresholds, precisions[:-1], 'b--', label='Precision')
plt.plot(thresholds, recalls[:-1], 'g-', label='Recall')
plt.plot(thresholds, f2_scores, 'r-', label='F2 Score', linewidth=2)

# Mark the optimal threshold
plt.axvline(x=optimal_p, color='k', linestyle=':')
plt.title('Threshold vs. Precision, Recall, and F2')
plt.xlabel('Probability Threshold')
plt.ylabel('Score')
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.show()

In [ ]:
model_data_filtered['fruiting_prob_pred'] = best_xgb_model.predict(model_data_filtered[final_features])
model_data_filtered['fruiting_pred'] = (model_data_filtered['fruiting_prob_pred'] > optimal_p).astype(int)

In [ ]:
model_data_filtered['fruiting_pred'].value_counts()
# model_data_filtered['train_test'].value_counts()

In [ ]:
# X = model_data_filtered[model_data_filtered['fruiting_pred'] == 1, model_features].copy()
# # X['ones'] = 1
# y = model_data_filtered[model_data_filtered['fruiting_pred'] == 1,'unique_users_edibles_smoothed'].copy()
# w = model_data_filtered[model_data_filtered['fruiting_pred'] == 1,'unique_users_fungi_smoothed'].copy()

X_train = model_data_filtered.loc[(model_data_filtered['fruiting_pred'] == 1) & (model_data_filtered['train_test'] == 'train'), final_features].copy()
X_test = model_data_filtered.loc[(model_data_filtered['fruiting_pred'] == 1) & (model_data_filtered['train_test'] == 'test'), final_features].copy()
y_train = model_data_filtered.loc[(model_data_filtered['fruiting_pred'] == 1) & (model_data_filtered['train_test'] == 'train'), 'unique_users_edibles_smoothed'].copy()
y_test = model_data_filtered.loc[(model_data_filtered['fruiting_pred'] == 1) & (model_data_filtered['train_test'] == 'test'), 'unique_users_edibles_smoothed'].copy()
w_train = model_data_filtered.loc[(model_data_filtered['fruiting_pred'] == 1) & (model_data_filtered['train_test'] == 'train'), 'unique_users_fungi_smoothed'].copy()
w_test = model_data_filtered.loc[(model_data_filtered['fruiting_pred'] == 1) & (model_data_filtered['train_test'] == 'test'), 'unique_users_fungi_smoothed'].copy()

In [ ]:
import numpy as np
import optuna
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from xgboost import XGBRegressor


def objective(trial):
    params = {
        'objective': 'reg:absoluteerror',
        'eval_metric': 'mae',
        'tree_method': 'hist',
        'learning_rate': trial.suggest_float(
            'learning_rate', 0.01, 0.1, log=True
        ),
        'max_depth': trial.suggest_int('max_depth', 3, 7),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 50.0, log=True),
        'n_estimators': 1500,
        'early_stopping_rounds': 30,
        'random_state': 42,
    }

    # Split by Year so entire seasons are held out
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_errors = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_va = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_va = y_train.iloc[train_idx], y_train.iloc[val_idx]
        w_tr, w_va = w_train.iloc[train_idx], w_train.iloc[val_idx]

        model = XGBRegressor(**params)
        model.fit(
            X_tr,
            y_tr,
            sample_weight=w_tr,
            eval_set=[(X_va, y_va)],
            sample_weight_eval_set=[w_va],
            verbose=False,
        )

        preds = model.predict(X_va)
        mae = mean_absolute_error(y_va, preds, sample_weight=w_va)
        fold_errors.append(mae)

    return np.mean(fold_errors)


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=40, show_progress_bar=True)

print("Best parameters for XGBoost:", study.best_params)

best_xgb_regressor = XGBRegressor(
    **study.best_params,
    random_state=42,
    objective='reg:squarederror',
    eval_metric='rmse'
)
best_xgb_regressor.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    # early_stopping_rounds=10,
    verbose=False
)

y_pred_xgb = best_xgb_regressor.predict(X_test)

mae_xgb = mean_absolute_error(y_test, y_pred_xgb, sample_weight=w_test)
mse_xgb = mean_squared_error(y_test, y_pred_xgb, sample_weight=w_test)
r2_xgb = r2_score(y_test, y_pred_xgb, sample_weight=w_test)

print(f"XGBoost MAE: {mae_xgb}")
print(f"XGBoost MSE: {mse_xgb}")
print(f"XGBoost R2: {r2_xgb}")



plt.scatter(y_test, y_pred_xgb)
plt.xlabel('Actual Count')
plt.ylabel('Predicted XGBoost Count')
plt.title('Actual vs Predicted Counts (XGBoost)')
plt.show()


In [ ]:
import xgboost as xgb

xgb.plot_importance(best_xgb_regressor, importance_type='gain', height=0.4, max_num_features=20)
plt.title('Top 20 Feature Importances (Gain)')
plt.show()

In [ ]:
model_data_filtered['pred_final'] = 0
model_data_filtered.loc[(model_data_filtered['fruiting_pred'] == 1), 'pred_final'] = best_xgb_regressor.predict(model_data_filtered.loc[(model_data_filtered['fruiting_pred'] == 1), final_features]) * model_data_filtered.loc[(model_data_filtered['fruiting_pred'] == 1), 'fruiting_prob_pred']
# model_data_filtered['pred_final'].value_counts()
# model_data_filtered['pred_final'].isna().sum()
# model_data_filtered['pred_final'].describe()

from sklearn.metrics import r2_score

# Calculate R2 score for train set
test_mask = model_data_filtered['train_test'] == 'test'
r2_pred_final_test = r2_score(
    model_data_filtered.loc[test_mask, 'unique_users_edibles_smoothed'],
    model_data_filtered.loc[test_mask, 'pred_final']
)
print(f"R2 score for pred_final vs. unique_users_edibles_smoothed (test set): {r2_pred_final_test}")



In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Create figure with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])


# Plot Precipitation (mm) - Open-Meteo
fig.add_trace(go.Bar(
    x=model_data_filtered['date'],
    y=model_data_filtered['prcp_mm'],
    name='Precipitation (mm) - Open-Meteo',
    # mode='markers',
    # marker_color='blue',
    # marker_size=1,
    opacity=1.0
))

fig.add_trace(go.Scatter(
    x=model_data_filtered['date'],
    y=model_data_filtered['unique_users_edibles_smoothed'],
    name='Users Reporting Edibles',
    mode='markers',
    marker_color='brown',
    marker_size=3,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

# Plot user counts from XGBoost model
# predX_xgb = model_data_filtered.copy()
# predX_xgb['pred_users_xgb'] = best_xgb_model.predict(predX_xgb[model_features])
# predX_xgb = predX_xgb.sort_values(by='date')
fig.add_trace(go.Scatter(
    x=model_data_filtered['date'],
    y=model_data_filtered['pred_final'],
    name='XGB Predicted Users Reporting Edibles',
    mode='lines+markers',
    marker_color='red',
    marker_size=4,
    line_width=1,
    marker_symbol='circle',
    opacity=1.0
    ),
    secondary_y=True
)

fig.update_layout(
    barmode='overlay',
    title='Num Users Reporting Edibles and Predictions by Date, with Precipitation',
    xaxis_title='Date',
    yaxis_title='Precip (mm)',
    yaxis2_title='Count users',
    legend=dict(x=0.01, y=0.99, bordercolor='gray', borderwidth=1),
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1y", step="year", stepmode="backward"),
                dict(count=3, label="3y", step="year", stepmode="backward"),
                dict(step="all")
            ])
        ),
        rangeslider=dict(
            visible=True
        ),
        type="date"
    ),
    height=800,
    template='plotly_white'
)

fig.show()